# 🌱 Multi-Scale Crop Health Classification
### Research-Grade Multi-Modal Training on Google Colab

This notebook provides a complete environment to train and evaluate the **Multi-Scale Crop Health Classification** system using GPU acceleration.

---

## 1. Environment Setup
First, we connect to a GPU and mount Google Drive to persist our code and data.

In [ ]:
# Check for GPU availability
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
if device != "cuda":
    print("WARNING: Training will be very slow without a GPU. Please go to 'Runtime' -> 'Change runtime type' and select T4 GPU.")

!nvidia-smi

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Define your project directory relative to your Google Drive
PROJ_DRIVE_PATH = '/content/drive/MyDrive/multi-scale-crop-health'
BASE_DIR = '/content/multi-scale-crop-health'

import os
os.makedirs(BASE_DIR, exist_ok=True)

## 2. Install Dependencies
We need specific versions of PyTorch and auxiliary libraries like `tifffile` and `grad-cam`.

In [ ]:
!pip install torch torchvision scikit-learn matplotlib seaborn pandas pyyaml tqdm opencv-python pillow grad-cam tifffile -q
print("Dependencies installed.")

## 3. Project Code Setup
If you have zipped your project locally, upload it to your Google Drive and set the path above. Alternatively, you can use `git clone` if your repo is public.

**Choose one of the following cells:**

In [ ]:
# OPTION A: If you uploaded your project folder to Google Drive
# !cp -r "$PROJ_DRIVE_PATH"/* "$BASE_DIR/"

# OPTION B: If you zipped your project folder (e.g. project.zip) and uploaded it to Drive
# !unzip "$PROJ_DRIVE_PATH/project.zip" -d "$BASE_DIR/"

print("Ensure your project files (models/, training/, etc.) are in:", BASE_DIR)
!ls $BASE_DIR

## 4. Dataset Preparation
Upload `plantvillage.zip`, `mhsoyahealthvision.zip`, and `eurosat.zip` to a `data_zips` folder on your Drive.

In [ ]:
DATA_ZIPS_PATH = '/content/drive/MyDrive/multi-scale-crop-health/data_zips' # Update this!

# Create data directory
os.makedirs(f"{BASE_DIR}/data/raw", exist_ok=True)

# Copy and extract datasets
!cp "$DATA_ZIPS_PATH/plantvillage.zip" "$BASE_DIR/"
!cp "$DATA_ZIPS_PATH/mhsoyahealthvision.zip" "$BASE_DIR/"
!cp "$DATA_ZIPS_PATH/eurosat.zip" "$BASE_DIR/"

import sys
sys.path.append(BASE_DIR)

# Run extraction script (Ensure scripts/extract_datasets.py is available)
!python "$BASE_DIR/scripts/extract_datasets.py"

# Run TIFF to JPG conversion for EuroSAT
!python "$BASE_DIR/scripts/convert_tif_to_jpg.py"

# Run Biological Alignment & Triplet Generation
!python "$BASE_DIR/preprocessing/align_datasets.py"

## 5. Training
We will now train the full fusion model. You can adjust the configuration using `configs/default_config.yaml` or by passing arguments.

In [ ]:
%cd $BASE_DIR

# Train Full Fusion Model
!python experiments/run_experiment.py --config configs/exp_full_fusion.yaml

## 6. Ablation Study (Optional)
Run all experiments sequentially to compare different modality combinations.

In [ ]:
!python experiments/run_experiment.py --ablation

## 7. Results & Download
Check the `outputs` directory for metrics and checkpoints. You can zip and download them to your local machine.

In [ ]:
import shutil
from google.colab import files

# Zip results
shutil.make_archive('/content/crop_health_results', 'zip', f'{BASE_DIR}/outputs')

# Download
files.download('/content/crop_health_results.zip')